# Bölüm 8: Doğrusal Regresyona Giriş
**Veri Madenciliği Dersi — Haydar Kılıç**

---

Bu notebook'ta şu konuları ele alacağız:
1. Doğru Uydurma, Artıklar ve Korelasyon
2. En Küçük Kareler Regresyonu ile Doğru Uydurma
3. Doğrusal Regresyonda Aykırı Değer Türleri
4. Doğrusal Regresyon için Çıkarım

In [ ]:
# Gerekli kütüphaneler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy.stats as stats
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Grafik stili
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Türkçe karakter desteği için
plt.rcParams['axes.unicode_minus'] = False

print("Kütüphaneler başarıyla yüklendi!")

---
## BÖLÜM 1: Doğru Uydurma, Artıklar ve Korelasyon

### 1.1 Veri Seti: Yoksulluk — Lise Mezunu Oranı

ABD'deki 50 eyalet + DC verisiyle çalışıyoruz.  
- **Yanıt değişkeni (y):** Yoksulluk oranı (%)  
- **Açıklayıcı değişken (x):** Lise mezunu oranı (%)

In [ ]:
# Sunumdaki özet istatistiklere dayalı sentetik veri
# x_bar=86.01, sx=3.73, y_bar=11.35, sy=3.1, R=-0.75
np.random.seed(42)
n = 51  # 50 eyalet + DC

x_bar, sx = 86.01, 3.73
y_bar, sy = 11.35, 3.10
R = -0.75

# Kovaryans matrisinden çift değişkenli normal dağılım üret
cov_xy = R * sx * sy
cov_matrix = [[sx**2, cov_xy], [cov_xy, sy**2]]
data = np.random.multivariate_normal([x_bar, y_bar], cov_matrix, n)

lise_mez = data[:, 0]
yoksulluk = data[:, 1]

# Gerçekçi aralıklara kırp
lise_mez = np.clip(lise_mez, 78, 93)
yoksulluk = np.clip(yoksulluk, 5.5, 19)

# DC ve RI'yı temsil eden özel noktalar
# DC: lise_mez=86, yoksulluk=16.8  (artık ≈ +5.44)
# RI: lise_mez=81, yoksulluk=10.3  (artık ≈ -4.16)
lise_mez[-2], yoksulluk[-2] = 86.0, 16.8   # DC
lise_mez[-1], yoksulluk[-1] = 81.0, 10.3   # RI

df = pd.DataFrame({'lise_mez': lise_mez, 'yoksulluk': yoksulluk})

print("Veri özeti:")
print(df.describe().round(2))
print(f"\nKorelasyon: {df['lise_mez'].corr(df['yoksulluk']):.3f}")

In [ ]:
# Saçılım grafiği
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(lise_mez, yoksulluk, color='steelblue', alpha=0.7, s=60, edgecolors='white')
ax.set_xlabel('Lise Mezunu Oranı (%)')
ax.set_ylabel('Yoksulluk Oranı (%)')
ax.set_title('Lise Mezunu Oranı ile Yoksulluk Oranı Arasındaki İlişki\n(ABD — 50 Eyalet + DC, 2012)')

# DC ve RI etiketle
ax.annotate('DC', xy=(86, 16.8), xytext=(84.5, 17.8),
            fontsize=10, color='darkred',
            arrowprops=dict(arrowstyle='->', color='darkred', lw=1.2))
ax.annotate('RI', xy=(81, 10.3), xytext=(79.5, 9.2),
            fontsize=10, color='darkgreen',
            arrowprops=dict(arrowstyle='->', color='darkgreen', lw=1.2))

plt.tight_layout()
plt.show()

print("Gözlem:")
print("  - İlişki: doğrusal, negatif, orta güçte")
print("  - Lise mezunu oranı arttıkça yoksulluk oranı azalmaktadır.")

---
### 1.2 Regresyon Modeli ve Tahmin

Sunumdaki model:
$$\widehat{\text{yoksulluk}} = 64{,}78 - 0{,}62 \times \text{lise\_mez}$$

"Şapka" (^) işareti bunun bir **tahmin** olduğunu gösterir.

In [ ]:
# Sunumdaki katsayılar
b0 = 64.78
b1 = -0.62

def tahmin_et(lise_mez_degeri):
    """Doğrusal model ile yoksulluk oranı tahmini"""
    return b0 + b1 * lise_mez_degeri

# Georgia örneği (sunumdan): lise_mez = %85.1
georgia_lise = 85.1
georgia_tahmin = tahmin_et(georgia_lise)
print(f"Georgia için tahmin (lise mezunu = {georgia_lise}%):")
print(f"  ŷ = {b0} + ({b1}) × {georgia_lise} = {georgia_tahmin:.3f}")
print()

# Birkaç farklı eyalet için tahmin tablosu
eyaletler = {
    'Georgia': 85.1,
    'Mississippi': 80.5,
    'Minnesota': 91.8,
    'Utah': 90.4,
    'DC': 86.0,
    'RI': 81.0
}

print(f"{'Eyalet':<12} {'Lise Mez. %':>12} {'Tahmin (ŷ)':>12}")
print("-" * 38)
for eyalet, lise in eyaletler.items():
    print(f"{eyalet:<12} {lise:>12.1f} {tahmin_et(lise):>12.2f}")

---
### 1.3 Artıklar (Residuals)

**Artık**, gözlemlenen $(y_i)$ ile tahmin edilen $(\hat{y}_i)$ arasındaki farktır:
$$e_i = y_i - \hat{y}_i$$

Temel eşitlik:
$$\text{Veri} = \text{Uyum} + \text{Artık}$$

In [ ]:
# Tüm veri için tahmin ve artık hesapla
y_hat = tahmin_et(lise_mez)
artiklar = yoksulluk - y_hat

# DC ve RI artıklarını hesapla
dc_lise, dc_yoks = 86.0, 16.8
ri_lise, ri_yoks = 81.0, 10.3

dc_yhat = tahmin_et(dc_lise)
ri_yhat = tahmin_et(ri_lise)
dc_artik = dc_yoks - dc_yhat
ri_artik = ri_yoks - ri_yhat

print("DC (Washington D.C.):")
print(f"  Gözlemlenen: {dc_yoks}")
print(f"  Tahmin:      {b0} - {abs(b1)} × {dc_lise} = {dc_yhat:.2f}")
print(f"  Artık:       {dc_yoks} - {dc_yhat:.2f} = {dc_artik:.2f}  ← tahmin EDİLDEKİNDEN {abs(dc_artik):.2f} FAZLA")
print()
print("RI (Rhode Island):")
print(f"  Gözlemlenen: {ri_yoks}")
print(f"  Tahmin:      {b0} - {abs(b1)} × {ri_lise} = {ri_yhat:.2f}")
print(f"  Artık:       {ri_yoks} - {ri_yhat:.2f} = {ri_artik:.2f}  ← tahmin edilenden {abs(ri_artik):.2f} AZ")

In [ ]:
# Artıkları görselleştir
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_line = np.linspace(77, 94, 200)
y_line = tahmin_et(x_line)

# Sol grafik: saçılım + regresyon doğrusu + artık çizgileri
ax = axes[0]
ax.scatter(lise_mez[:-2], yoksulluk[:-2], color='steelblue', alpha=0.6, s=50, zorder=3)
ax.plot(x_line, y_line, 'r-', linewidth=2.5, label='Regresyon doğrusu', zorder=2)

# Artık çizgileri
for i in range(len(lise_mez)-2):
    color = 'green' if artiklar[i] > 0 else 'darkgreen'
    ax.plot([lise_mez[i], lise_mez[i]], [yoksulluk[i], y_hat[i]],
            color='olive', alpha=0.5, linewidth=1, zorder=1)

# DC ve RI
ax.scatter([dc_lise, ri_lise], [dc_yoks, ri_yoks],
           color=['darkred', 'darkgreen'], s=80, zorder=5)
ax.plot([dc_lise, dc_lise], [dc_yhat, dc_yoks], 'r-', linewidth=2)
ax.plot([ri_lise, ri_lise], [ri_yhat, ri_yoks], 'g-', linewidth=2)
ax.annotate(f'DC\n+{dc_artik:.2f}', xy=(dc_lise+0.2, (dc_yoks+dc_yhat)/2),
            color='darkred', fontsize=9, fontweight='bold')
ax.annotate(f'RI\n{ri_artik:.2f}', xy=(ri_lise-2.2, (ri_yoks+ri_yhat)/2),
            color='darkgreen', fontsize=9, fontweight='bold')

ax.set_xlabel('Lise Mezunu Oranı (%)')
ax.set_ylabel('Yoksulluk Oranı (%)')
ax.set_title('Regresyon Doğrusu ve Artıklar')
ax.legend()

# Sağ grafik: artık grafiği
ax2 = axes[1]
ax2.scatter(lise_mez[:-2], artiklar[:-2], color='steelblue', alpha=0.6, s=50)
ax2.scatter([dc_lise, ri_lise], [dc_artik, ri_artik],
            color=['darkred', 'darkgreen'], s=80, zorder=5)
ax2.axhline(0, color='red', linestyle='--', linewidth=1.5)
ax2.set_xlabel('Lise Mezunu Oranı (%)')
ax2.set_ylabel('Artık (Gözlem − Tahmin)')
ax2.set_title('Artıklar Grafiği')
ax2.annotate('DC', xy=(dc_lise+0.2, dc_artik), color='darkred', fontweight='bold')
ax2.annotate('RI', xy=(ri_lise+0.2, ri_artik), color='darkgreen', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nArtıkların ortalaması (≈ 0 olmalı): {artiklar.mean():.4f}")
print(f"Artıkların standart sapması: {artiklar.std():.4f}")

---
### 1.4 Korelasyon

**Korelasyon**, iki değişken arasındaki **doğrusal ilişkinin gücünü** tanımlar.  
- $-1$ (mükemmel negatif) ile $+1$ (mükemmel pozitif) arasında değerler alır  
- $0$ değeri doğrusal ilişki olmadığını gösterir

> ⚠️ Korelasyon yalnızca **doğrusal** ilişkileri ölçer!

In [ ]:
# Farklı korelasyon değerlerini görselleştir
np.random.seed(0)
n_demo = 80

korelasyonlar = {
    'R ≈ +0.95 (Güçlü Pozitif)': 0.95,
    'R ≈ +0.50 (Orta Pozitif)': 0.50,
    'R ≈ 0.00 (İlişki Yok)': 0.00,
    'R ≈ -0.75 (Orta Negatif)': -0.75,
    'R ≈ -0.95 (Güçlü Negatif)': -0.95,
    'R = ? (Doğrusal Olmayan)': None
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (baslik, r) in enumerate(korelasyonlar.items()):
    ax = axes[idx]
    if r is not None:
        cov = [[1, r], [r, 1]]
        xy = np.random.multivariate_normal([0, 0], cov, n_demo)
        x_d, y_d = xy[:, 0], xy[:, 1]
        r_calc = np.corrcoef(x_d, y_d)[0, 1]
        label = f'R = {r_calc:.2f}'
    else:
        # Doğrusal olmayan: U şekli
        x_d = np.linspace(-3, 3, n_demo) + np.random.normal(0, 0.3, n_demo)
        y_d = x_d**2 + np.random.normal(0, 0.5, n_demo)
        r_calc = np.corrcoef(x_d, y_d)[0, 1]
        label = f'R = {r_calc:.2f} (U şekli!)'

    ax.scatter(x_d, y_d, alpha=0.5, s=25, color='steelblue')
    ax.set_title(baslik, fontsize=10)
    ax.set_xlabel(label, fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle('Farklı Korelasyon Değerlerinin Görselleştirilmesi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("NOT: Sağ alt grafik (U şekli) — Aralarında güçlü bir ilişki olmasına rağmen")
print("korelasyon ≈ 0 çıkıyor. Korelasyon sadece DOĞRUSAL ilişkiyi ölçer!")

---
## BÖLÜM 2: En Küçük Kareler Regresyonu ile Doğru Uydurma

### 2.1 En Küçük Kareler Yöntemi

**Amaç:** Karesel artıklar toplamını minimize eden doğruyu bul:
$$\text{Minimize et: } e_1^2 + e_2^2 + \cdots + e_n^2 = \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

**Neden karesel?**
- En yaygın kullanılan yöntem
- Elle ve yazılımla hesaplaması daha kolay
- Büyük artıklara daha fazla ceza verir

In [ ]:
# En küçük kareler formülleri
# b1 = (sy / sx) * R
# b0 = y_bar - b1 * x_bar

sx_calc = np.std(lise_mez, ddof=1)
sy_calc = np.std(yoksulluk, ddof=1)
R_calc  = np.corrcoef(lise_mez, yoksulluk)[0, 1]
x_bar_calc = np.mean(lise_mez)
y_bar_calc = np.mean(yoksulluk)

b1_calc = (sy_calc / sx_calc) * R_calc
b0_calc = y_bar_calc - b1_calc * x_bar_calc

print("Özet İstatistikler:")
print(f"  x̄  (lise mez. ortalama) = {x_bar_calc:.2f}")
print(f"  ȳ  (yoksulluk ortalama) = {y_bar_calc:.2f}")
print(f"  sx (lise mez. st. sapma)= {sx_calc:.2f}")
print(f"  sy (yoksulluk st. sapma)= {sy_calc:.2f}")
print(f"  R  (korelasyon)         = {R_calc:.3f}")
print()
print("Eğim Hesabı:")
print(f"  b1 = (sy/sx) × R = ({sy_calc:.2f}/{sx_calc:.2f}) × ({R_calc:.3f}) = {b1_calc:.4f}")
print()
print("Kesim Noktası Hesabı:")
print(f"  b0 = ȳ - b1×x̄ = {y_bar_calc:.2f} - ({b1_calc:.4f})×{x_bar_calc:.2f} = {b0_calc:.4f}")
print()
print(f"Model: ŷ = {b0_calc:.2f} + ({b1_calc:.2f}) × x")
print(f"Sunumdaki model: ŷ = 64.78 - 0.62 × x  (uyuşuyor ✓)")

In [ ]:
# sklearn ile doğrulama
model = LinearRegression()
model.fit(lise_mez.reshape(-1, 1), yoksulluk)

print("sklearn LinearRegression sonuçları:")
print(f"  b0 (kesim) = {model.intercept_:.4f}")
print(f"  b1 (eğim)  = {model.coef_[0]:.4f}")
print(f"  R²         = {model.score(lise_mez.reshape(-1,1), yoksulluk):.4f}")

---
### 2.2 Model Katsayılarının Yorumu

$$\widehat{\text{yoksulluk}} = 64{,}78 - 0{,}62 \times \text{lise\_mez}$$

- **Eğim (b₁ = -0.62):** Lise mezunu oranındaki her ek **%1** artış için, yoksulluk oranının ortalama olarak **0.62 puan** daha düşük olması beklenir.
- **Kesim noktası (b₀ = 64.78):** Lise mezunu oranı **%0** olan bir eyalette (gerçekte yok — dışavurum!), yoksulluk oranının %64.78 olması beklenir.

> ⚠️ Bu ifadeler, çalışma rastgele kontrollü bir deney olmadıkça **nedensel değildir**!

In [ ]:
# Regresyon doğrusunu ve kesim noktasını görselleştir
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Veri aralığında model
ax1 = axes[0]
x_range = np.linspace(77, 94, 300)
y_range = b0 + b1 * x_range
ax1.scatter(lise_mez, yoksulluk, color='steelblue', alpha=0.6, s=55, zorder=3)
ax1.plot(x_range, y_range, 'r-', linewidth=2.5)

# Ortalama noktayı vurgula — regresyon her zaman (x̄, ȳ)'den geçer!
ax1.scatter([x_bar_calc], [y_bar_calc], color='orange', s=150, zorder=5,
            marker='*', label=f'(x̄={x_bar_calc:.1f}, ȳ={y_bar_calc:.1f})')
ax1.axvline(x_bar_calc, color='orange', linestyle=':', alpha=0.5)
ax1.axhline(y_bar_calc, color='orange', linestyle=':', alpha=0.5)
ax1.set_xlabel('Lise Mezunu Oranı (%)')
ax1.set_ylabel('Yoksulluk Oranı (%)')
ax1.set_title('Regresyon Doğrusu (Veri Aralığı)')
ax1.legend()
ax1.text(88, 17, f'ŷ = {b0:.2f} + ({b1})×x', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Sağ: Tam aralık — dışavurum tehlikesi
ax2 = axes[1]
x_full = np.linspace(0, 100, 300)
y_full = b0 + b1 * x_full
ax2.scatter(lise_mez, yoksulluk, color='steelblue', alpha=0.6, s=55)
ax2.plot(x_full, y_full, 'r-', linewidth=2)
ax2.axvline(x=0, color='purple', linestyle='--', alpha=0.5)
ax2.scatter([0], [b0], color='purple', s=120, zorder=5, label=f'Kesim noktası = {b0}')
ax2.fill_betweenx([-10, 80], 0, 75, alpha=0.05, color='red', label='Dışavurum bölgesi')
ax2.set_xlim(-2, 103)
ax2.set_ylim(-5, 72)
ax2.set_xlabel('Lise Mezunu Oranı (%)')
ax2.set_ylabel('Yoksulluk Oranı (%)')
ax2.set_title('Tam Aralık — Dışavurum Tehlikesi!')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("⚠️  Kesim noktası (x=0): Veri setinde lise mezunu olmayan eyalet yok!")
print("    Bu nedenle kesim noktasının yorumlanması güvenilir değildir (dışavurum).")

---
### 2.3 Model Koşulları

En küçük kareler doğrusu için **3 temel koşul** gereklidir:

| Koşul | Kontrol Yöntemi |
|---|---|
| 1. Doğrusallık | Saçılım grafiği veya artıklar grafiği |
| 2. Yaklaşık normal artıklar | Histogram |
| 3. Sabit değişkenlik (Homoskedastisite) | Artıklar grafiği |

In [ ]:
# 3 koşulun tam tanı grafikleri
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

y_pred = tahmin_et(lise_mez)
residuals = yoksulluk - y_pred

# ---- KOŞUL 1: DOĞRUSALLİK ----
# İyi örnek
ax = axes[0, 0]
ax.scatter(lise_mez, yoksulluk, color='steelblue', alpha=0.6, s=40)
ax.plot(np.linspace(77, 94, 100), tahmin_et(np.linspace(77, 94, 100)), 'r-', lw=2)
ax.set_title('Koşul 1 — Doğrusallık ✓\n(İyi Örnek)', color='green', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('y')

# Kötü örnek — eğrisel ilişki
ax = axes[1, 0]
x_bad = np.linspace(-3, 3, 60) + np.random.normal(0, 0.2, 60)
y_bad = x_bad**2 + np.random.normal(0, 0.3, 60)
m_bad = LinearRegression().fit(x_bad.reshape(-1,1), y_bad)
ax.scatter(x_bad, y_bad, color='steelblue', alpha=0.6, s=40)
x_s = np.linspace(-3.5, 3.5, 100)
ax.plot(x_s, m_bad.predict(x_s.reshape(-1,1)), 'r-', lw=2)
res_bad = y_bad - m_bad.predict(x_bad.reshape(-1,1))
ax.set_title('Koşul 1 — Doğrusallık ✗\n(Artık grafiğinde eğrilik)', color='red', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('Artık')

# ---- KOŞUL 2: NORMAL ARTIKLAR ----
ax = axes[0, 1]
ax.hist(residuals, bins=12, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='red', linestyle='--')
ax.set_title('Koşul 2 — Yaklaşık Normal Artıklar ✓\n(Histogram)', color='green', fontsize=10)
ax.set_xlabel('Artık'); ax.set_ylabel('Frekans')

# Kötü örnek — çarpık dağılım
ax = axes[1, 1]
skewed_res = np.random.exponential(scale=1.5, size=100) - 1.5
ax.hist(skewed_res, bins=12, color='salmon', edgecolor='white', alpha=0.8)
ax.axvline(0, color='red', linestyle='--')
ax.set_title('Koşul 2 — Normal Artıklar ✗\n(Çarpık dağılım)', color='red', fontsize=10)
ax.set_xlabel('Artık'); ax.set_ylabel('Frekans')

# ---- KOŞUL 3: SABİT DEĞİŞKENLİK ----
ax = axes[0, 2]
ax.scatter(y_pred, residuals, color='steelblue', alpha=0.6, s=40)
ax.axhline(0, color='red', linestyle='--')
ax.set_title('Koşul 3 — Sabit Değişkenlik ✓\n(Homoskedastisite)', color='green', fontsize=10)
ax.set_xlabel('Tahmin (ŷ)'); ax.set_ylabel('Artık')

# Kötü örnek — huni şekli
ax = axes[1, 2]
x_fan = np.linspace(1, 10, 80)
y_fan = 2 * x_fan + np.random.normal(0, x_fan * 0.5, 80)
m_fan = LinearRegression().fit(x_fan.reshape(-1,1), y_fan)
res_fan = y_fan - m_fan.predict(x_fan.reshape(-1,1))
ax.scatter(m_fan.predict(x_fan.reshape(-1,1)), res_fan, color='salmon', alpha=0.6, s=40)
ax.axhline(0, color='red', linestyle='--')
ax.set_title('Koşul 3 — Sabit Değişkenlik ✗\n(Huni şekli — Heteroskedastisite)', color='red', fontsize=10)
ax.set_xlabel('Tahmin (ŷ)'); ax.set_ylabel('Artık')

plt.suptitle('En Küçük Kareler Koşullarının Tanı Grafikleri\n(Yeşil: koşul sağlanıyor | Kırmızı: koşul ihlali)', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
### 2.4 R² — Belirleme Katsayısı

$$R^2 = r^2 = (-0{,}75)^2 = 0{,}56$$

**Yorumu:** Yanıt değişkenindeki değişkenliğin **%56'sı** model tarafından açıklanmaktadır.

In [ ]:
# R² hesabı — iki yöntem
R_korelasyon = np.corrcoef(lise_mez, yoksulluk)[0, 1]
R2_korelasyon = R_korelasyon**2

SS_tot = np.sum((yoksulluk - np.mean(yoksulluk))**2)
SS_res = np.sum((yoksulluk - y_pred)**2)
R2_direkt = 1 - SS_res / SS_tot

print("R² Hesabı:")
print(f"  Yöntem 1 — Korelasyon karesi: R² = ({R_korelasyon:.4f})² = {R2_korelasyon:.4f}")
print(f"  Yöntem 2 — SS formülü:         R² = 1 - SS_res/SS_tot = {R2_direkt:.4f}")
print()
print(f"  Sunumdaki değer: R² = (-0.75)² = 0.5625")
print()
print("Yorum:")
print(f"  51 eyaletteki yoksulluk oranındaki değişkenliğin %{R2_direkt*100:.1f}'si")
print(f"  lise mezunu oranını açıklayıcı değişken olarak kullanan model tarafından açıklanmaktadır.")
print(f"  Geri kalan %{(1-R2_direkt)*100:.1f} modele dahil edilmeyen değişkenler veya")
print(f"  verideki doğal rastgelelik tarafından açıklanır.")

In [ ]:
# R² görselleştirmesi: Toplam varyans vs. açıklanan varyans
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sol: Model olmadan (sadece ortalama)
ax1 = axes[0]
ax1.scatter(lise_mez, yoksulluk, color='steelblue', alpha=0.5, s=40)
ax1.axhline(np.mean(yoksulluk), color='gray', linestyle='--', linewidth=2,
            label=f'ȳ = {np.mean(yoksulluk):.2f} (ortalama model)')
for i in range(len(lise_mez)):
    ax1.plot([lise_mez[i], lise_mez[i]], [yoksulluk[i], np.mean(yoksulluk)],
             color='gray', alpha=0.3, linewidth=0.8)
ax1.set_title(f'Toplam Değişkenlik (SS_tot)\nR² = 0 → hiçbir şey açıklanamıyor')
ax1.set_xlabel('Lise Mezunu Oranı (%)'); ax1.set_ylabel('Yoksulluk (%)')
ax1.legend(fontsize=9)

# Sağ: Model ile (regresyon doğrusu)
ax2 = axes[1]
ax2.scatter(lise_mez, yoksulluk, color='steelblue', alpha=0.5, s=40)
ax2.plot(x_range, tahmin_et(x_range), 'r-', linewidth=2, label='Regresyon doğrusu')
for i in range(len(lise_mez)):
    ax2.plot([lise_mez[i], lise_mez[i]], [yoksulluk[i], y_pred[i]],
             color='green', alpha=0.3, linewidth=0.8)
ax2.set_title(f'Açıklanamayan Değişkenlik (SS_res)\nR² = {R2_direkt:.2f} → %{R2_direkt*100:.0f} açıklandı')
ax2.set_xlabel('Lise Mezunu Oranı (%)'); ax2.set_ylabel('Yoksulluk (%)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## BÖLÜM 3: Aykırı Değer Türleri

Doğrusal regresyonda üç tür özel nokta vardır:

| Terim | Tanım |
|---|---|
| **Aykırı değer** | Nokta bulutundan uzakta konumlanan nokta |
| **Yüksek kaldıraçlı nokta** | Bulutun merkezinden yatay olarak uzakta olan nokta |
| **Etkili nokta** | Regresyon doğrusunun eğimini gerçekten değiştiren yüksek kaldıraçlı nokta |

In [ ]:
# Aykırı değer türlerini karşılaştır
np.random.seed(7)
n_ayk = 40

x_base = np.random.normal(5, 1, n_ayk)
y_base = 2 * x_base + np.random.normal(0, 1, n_ayk)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

senaryolar = [
    {
        'baslik': 'Aykırı Değer\n(Yüksek Artık, Düşük Kaldıraç)',
        'x_out': 5.0, 'y_out': 20.0,
        'renk': 'red',
        'aciklama': 'x ekseni merkezinde\nama y ekseni çok yüksek'
    },
    {
        'baslik': 'Yüksek Kaldıraçlı + Etkili\n(Eğimi dramatik değiştirir)',
        'x_out': 13.0, 'y_out': 8.0,
        'renk': 'darkred',
        'aciklama': 'x ekseni uzağında\nVE modelden sapıyor'
    },
    {
        'baslik': 'Yüksek Kaldıraçlı + Etkisiz\n(Trendi takip ediyor)',
        'x_out': 13.0, 'y_out': 26.0,
        'renk': 'orange',
        'aciklama': 'x ekseni uzağında\nAMA trend üzerinde'
    }
]

for idx, senaryo in enumerate(senaryolar):
    ax = axes[idx]
    x_ayk = np.append(x_base, senaryo['x_out'])
    y_ayk = np.append(y_base, senaryo['y_out'])

    # Aykırı değer olmadan model
    m_no  = LinearRegression().fit(x_base.reshape(-1,1), y_base)
    # Aykırı değer ile model
    m_yes = LinearRegression().fit(x_ayk.reshape(-1,1), y_ayk)

    x_plt = np.linspace(min(x_ayk)-0.5, max(x_ayk)+0.5, 200)

    ax.scatter(x_base, y_base, color='steelblue', alpha=0.5, s=35, label='Normal noktalar')
    ax.scatter([senaryo['x_out']], [senaryo['y_out']],
               color=senaryo['renk'], s=120, zorder=5,
               marker='D', label='Aykırı nokta', edgecolors='black')

    ax.plot(x_plt, m_no.predict(x_plt.reshape(-1,1)),
            'g--', linewidth=2, label=f'Aykırı yok (b₁={m_no.coef_[0]:.2f})')
    ax.plot(x_plt, m_yes.predict(x_plt.reshape(-1,1)),
            'r-', linewidth=2, label=f'Aykırı var (b₁={m_yes.coef_[0]:.2f})')

    degisim = abs(m_yes.coef_[0] - m_no.coef_[0])
    ax.set_title(senaryo['baslik'], fontsize=9)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=7.5)
    ax.text(0.05, 0.95, f"Eğim değişimi: {degisim:.2f}",
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.suptitle('Aykırı Değer Türlerinin Regresyon Doğrusuna Etkisi', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Sonuç:")
print("  Senaryo 1: Artık büyük ama eğimi fazla değiştirmez → 'aykırı değer, düşük kaldıraç'")
print("  Senaryo 2: Hem uzak hem de eğimi dramatik değiştirir → 'etkili nokta'")
print("  Senaryo 3: Uzak ama trendi takip eder → 'yüksek kaldıraçlı ama etkisiz'")

---
## BÖLÜM 4: Doğrusal Regresyon için Çıkarım

### 4.1 Hipotez Testi — Eğimi Test Etmek

**Hipotezler:**
$$H_0: \beta_1 = 0 \quad (\text{ilişki yok})$$
$$H_A: \beta_1 \neq 0 \quad (\text{anlamlı bir ilişki var})$$

**Test istatistiği:**
$$T = \frac{b_1 - 0}{SH_{b_1}}, \quad df = n - 2$$

In [ ]:
# Sunumdaki doğa/yetiştirme (IQ) verisi
np.random.seed(15)
n_iq = 27

r_iq = 0.882
bio_iq = np.random.normal(100, 15, n_iq)
foster_noise = np.random.normal(0, np.sqrt(1 - r_iq**2) * 15, n_iq)
foster_iq = r_iq * (bio_iq - 100) + 100 + foster_noise
foster_iq = np.clip(foster_iq, 68, 135)

# sklearn ile fit
model_iq = LinearRegression()
model_iq.fit(bio_iq.reshape(-1, 1), foster_iq)

# Standart hata ve t-istatistiği manuel hesabı
y_pred_iq = model_iq.predict(bio_iq.reshape(-1, 1))
SS_res_iq = np.sum((foster_iq - y_pred_iq)**2)
df_iq = n_iq - 2
MSE_iq = SS_res_iq / df_iq
SE_b1_iq = np.sqrt(MSE_iq / np.sum((bio_iq - bio_iq.mean())**2))
SE_b0_iq = np.sqrt(MSE_iq * (1/n_iq + bio_iq.mean()**2 / np.sum((bio_iq - bio_iq.mean())**2)))

b1_iq = model_iq.coef_[0]
b0_iq = model_iq.intercept_

t_stat_b1 = b1_iq / SE_b1_iq
p_value_b1 = 2 * (1 - stats.t.cdf(abs(t_stat_b1), df=df_iq))

t_stat_b0 = b0_iq / SE_b0_iq
p_value_b0 = 2 * (1 - stats.t.cdf(abs(t_stat_b0), df=df_iq))

R2_iq = model_iq.score(bio_iq.reshape(-1, 1), foster_iq)

print("Regresyon Çıktısı — IQ Verisi (Doğa mı, Yetiştirme mi?)")
print("="*65)
print(f"{'':15} {'Tahmin':>10} {'Std. Hata':>10} {'t değeri':>10} {'p değeri':>10}")
print("-"*65)
print(f"{'(Kesim)':15} {b0_iq:>10.4f} {SE_b0_iq:>10.4f} {t_stat_b0:>10.2f} {p_value_b0:>10.4f}")
print(f"{'biyoIQ':15} {b1_iq:>10.4f} {SE_b1_iq:>10.4f} {t_stat_b1:>10.2f} {p_value_b1:>10.6f}")
print("="*65)
print(f"Artık standart hata: {np.sqrt(MSE_iq):.3f}, {df_iq} serbestlik derecesi")
print(f"R-kare: {R2_iq:.4f}")
print()
print(f"Sunumdaki değerler:  b0≈9.21 (SH≈9.30), b1≈0.90 (SH≈0.096), t≈9.36")
print()
print(f"Sonuç: p-değeri ({'<0.0001' if p_value_b1 < 0.0001 else f'{p_value_b1:.4f}'}) << 0.05")
print(f"→ H₀ reddedilir. Biyolojik IQ, koruyucu ailedeki IQ'nun anlamlı yordayıcısıdır.")

In [ ]:
# t dağılımı üzerinde test istatistiğini görselleştir
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sol: t dağılımı ve kritik bölge
ax1 = axes[0]
t_range = np.linspace(-12, 12, 500)
t_pdf = stats.t.pdf(t_range, df=df_iq)
t_crit = stats.t.ppf(0.975, df=df_iq)

ax1.plot(t_range, t_pdf, 'b-', linewidth=2, label=f't dağılımı (df={df_iq})')
ax1.fill_between(t_range, t_pdf, where=(t_range >= t_crit), color='red', alpha=0.3, label=f'Kritik bölge (α=0.05)')
ax1.fill_between(t_range, t_pdf, where=(t_range <= -t_crit), color='red', alpha=0.3)
ax1.axvline(t_stat_b1, color='darkred', linestyle='-', linewidth=2.5,
            label=f'T = {t_stat_b1:.2f} (gözlemlenen)')
ax1.axvline(t_crit, color='orange', linestyle='--', linewidth=1.5,
            label=f't* = ±{t_crit:.2f}')
ax1.set_xlim(-12, 12)
ax1.set_xlabel('t değeri')
ax1.set_ylabel('Yoğunluk')
ax1.set_title('Hipotez Testi: H₀: β₁ = 0')
ax1.legend(fontsize=8)
ax1.text(t_stat_b1 - 3.5, 0.15, f'p < 0.0001\nH₀ Reddedildi!',
         fontsize=10, color='darkred', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightyellow'))

# Sağ: Regresyon doğrusu
ax2 = axes[1]
ax2.scatter(bio_iq, foster_iq, color='steelblue', alpha=0.6, s=60)
x_iq_range = np.linspace(65, 140, 200)
ax2.plot(x_iq_range, model_iq.predict(x_iq_range.reshape(-1,1)),
         'r-', linewidth=2.5)
ax2.text(0.05, 0.92,
         f'ŷ = {b0_iq:.2f} + {b1_iq:.2f}×biyoIQ\nR = {np.sqrt(R2_iq):.3f}, R² = {R2_iq:.4f}',
         transform=ax2.transAxes, fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
ax2.set_xlabel('Biyolojik İkiz IQ')
ax2.set_ylabel('Koruyucu Ailede Büyüyen İkiz IQ')
ax2.set_title('Doğa mı, Yetiştirme mi?\n(IQ — Tek Yumurta İkizleri)')

plt.tight_layout()
plt.show()

---
### 4.2 Güven Aralığı — Eğim için

$$b_1 \pm t^*_{df=n-2} \times SH_{b_1}$$

Sunumdaki örnek için (n=27, df=25):
$$0{,}9014 \pm 2{,}06 \times 0{,}0963 = (0{,}70 \ ; \ 1{,}10)$$

In [ ]:
# Güven aralığı hesapla ve görselleştir
alpha_levels = [0.10, 0.05, 0.01]
konfidans_levels = [0.90, 0.95, 0.99]

print("Eğim (β₁) için Güven Aralıkları:")
print(f"  b₁ = {b1_iq:.4f}, SH = {SE_b1_iq:.4f}, df = {df_iq}")
print()
print(f"{'Güven Düzeyi':>15} {'t* değeri':>12} {'Alt Sınır':>12} {'Üst Sınır':>12}")
print("-" * 55)
for alpha, conf in zip(alpha_levels, konfidans_levels):
    t_crit_conf = stats.t.ppf(1 - alpha/2, df=df_iq)
    lower = b1_iq - t_crit_conf * SE_b1_iq
    upper = b1_iq + t_crit_conf * SE_b1_iq
    print(f"{conf*100:>14.0f}% {t_crit_conf:>12.3f} {lower:>12.4f} {upper:>12.4f}")

print()
t_star_25 = stats.t.ppf(0.975, df=25)
b1_sun = 0.9014
se_sun = 0.0963
lower_sun = b1_sun - t_star_25 * se_sun
upper_sun = b1_sun + t_star_25 * se_sun
print(f"Sunumdaki değerlerle: {b1_sun} ± {t_star_25:.2f} × {se_sun}")
print(f"  95% GA: ({lower_sun:.4f} , {upper_sun:.4f})  ≈ (0.70 , 1.10) ✓")

In [ ]:
# Güven aralıklarını çubuk grafikiyle görselleştir
fig, ax = plt.subplots(figsize=(9, 4))

renkler = ['#2196F3', '#4CAF50', '#F44336']
y_positions = [0, 1, 2]

for i, (alpha, conf) in enumerate(zip(alpha_levels, konfidans_levels)):
    t_c = stats.t.ppf(1 - alpha/2, df=df_iq)
    lower = b1_iq - t_c * SE_b1_iq
    upper = b1_iq + t_c * SE_b1_iq
    ax.barh(y_positions[i], upper - lower, left=lower, height=0.4,
            color=renkler[i], alpha=0.6, label=f'%{conf*100:.0f} GA')
    ax.scatter([b1_iq], [y_positions[i]], color='black', s=60, zorder=5)
    ax.text(upper + 0.01, y_positions[i], f'({lower:.3f}, {upper:.3f})',
            va='center', fontsize=9)

ax.axvline(0, color='red', linestyle='--', linewidth=1.5,
           label='β₁=0 (H₀)')
ax.set_yticks(y_positions)
ax.set_yticklabels(['%90 GA', '%95 GA', '%99 GA'])
ax.set_xlabel('β₁ (Eğim)')
ax.set_title('Eğim İçin Güven Aralıkları\n(Siyah nokta: nokta tahmini)')
ax.legend()
plt.tight_layout()
plt.show()

print("\nNOT: H₀: β₁=0 çizgisi hiçbir güven aralığının içinde değil.")
print("→ Tüm güven düzeylerinde H₀ reddedilir.")

---
## BÖLÜM 5: Ekstra — Dışavurum (Extrapolation)

Bir model tahminini **orijinal verilerin kapsamı dışındaki değerlere** uygulamaya **dışavurum** denir.

In [ ]:
# Dışavurum örneği: Evlilik yaşı zaman serisi
np.random.seed(20)
years_early = np.array([1900, 1910, 1920, 1930, 1940])
ages_early  = np.array([26.2, 26.1, 25.1, 24.6, 23.9])

model_marriage = LinearRegression()
model_marriage.fit(years_early.reshape(-1,1), ages_early)

years_recent = np.arange(1950, 1985, 2)
ages_recent  = 22.5 + 0.08 * (years_recent - 1960) + np.random.normal(0, 0.2, len(years_recent))

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(years_early,  ages_early,  color='steelblue', s=80, zorder=5,
           label='Erken dönem veri (1900–1940)')
ax.scatter(years_recent, ages_recent, color='darkred', s=50, zorder=5,
           label='Gerçek değerler (1950–1980)')

x_extrap = np.arange(1895, 1985, 1)
ax.plot(x_extrap, model_marriage.predict(x_extrap.reshape(-1,1)),
        'b-', linewidth=2, label='Modelin dışavurumu')
ax.axvspan(1945, 1985, alpha=0.1, color='red', label='Dışavurum bölgesi')
ax.axvline(1945, color='gray', linestyle='--', alpha=0.7)

ax.set_xlabel('Yıl')
ax.set_ylabel('İlk Evlilik Yaşı (Erkekler)')
ax.set_title('Dışavurum Tehlikesi: Evlilik Yaşı\n1900–1940 verisine dayalı model 1950 sonrasını yanlış tahmin ediyor')
ax.legend(fontsize=9)
ax.set_ylim(20, 28)
plt.tight_layout()
plt.show()

pred_1970 = model_marriage.predict([[1970]])[0]
print(f"Model 1970 tahmini: {pred_1970:.1f} yaş")
print(f"Gerçek 1970 değeri: ~23.2 yaş")
print(f"Hata: {abs(pred_1970 - 23.2):.1f} yıl — dışavurum ne kadar yanıltıcı olabilir!")

---
## BÖLÜM 6: Özet ve Kontrol Listesi

Bu derste öğrendiklerimiz:

In [ ]:
# Tam bir doğrusal regresyon analizi — tüm adımlar
print("=" * 60)
print("DOĞRUSAL REGRESYON ANALİZİ — TAM ADIMLAR")
print("=" * 60)

print("\n[ADIM 1] Özet İstatistikler")
x_bar_f = np.mean(lise_mez)
y_bar_f = np.mean(yoksulluk)
sx_f    = np.std(lise_mez, ddof=1)
sy_f    = np.std(yoksulluk, ddof=1)
R_f     = np.corrcoef(lise_mez, yoksulluk)[0,1]
print(f"  x̄ = {x_bar_f:.2f},  sx = {sx_f:.2f}")
print(f"  ȳ = {y_bar_f:.2f},  sy = {sy_f:.2f}")
print(f"  Korelasyon R = {R_f:.3f}")

print("\n[ADIM 2] Katsayılar")
b1_f = (sy_f / sx_f) * R_f
b0_f = y_bar_f - b1_f * x_bar_f
print(f"  Eğim:  b₁ = (sy/sx)×R = ({sy_f:.2f}/{sx_f:.2f})×{R_f:.3f} = {b1_f:.4f}")
print(f"  Kesim: b₀ = ȳ - b₁×x̄ = {y_bar_f:.2f} - ({b1_f:.4f})×{x_bar_f:.2f} = {b0_f:.4f}")
print(f"  Model: ŷ = {b0_f:.2f} + ({b1_f:.2f})×x")

print("\n[ADIM 3] Model Uyumu")
y_pred_f = b0_f + b1_f * lise_mez
SS_res_f = np.sum((yoksulluk - y_pred_f)**2)
SS_tot_f = np.sum((yoksulluk - y_bar_f)**2)
R2_f     = 1 - SS_res_f / SS_tot_f
RMSE_f   = np.sqrt(SS_res_f / (len(lise_mez)-2))
print(f"  R² = {R2_f:.4f} → Değişkenliğin %{R2_f*100:.1f}'i açıklandı")
print(f"  RMSE = {RMSE_f:.4f} (artık standart hata)")

print("\n[ADIM 4] Çıkarım (Eğim için)")
SE_b1_f = RMSE_f / np.sqrt(np.sum((lise_mez - x_bar_f)**2))
T_f     = b1_f / SE_b1_f
df_f    = len(lise_mez) - 2
p_f     = 2 * (1 - stats.t.cdf(abs(T_f), df=df_f))
t_star_f = stats.t.ppf(0.975, df=df_f)
ci_low   = b1_f - t_star_f * SE_b1_f
ci_high  = b1_f + t_star_f * SE_b1_f
print(f"  SH_b₁ = {SE_b1_f:.4f}")
print(f"  T = {T_f:.4f},  df = {df_f},  p-değeri = {p_f:.6f}")
print(f"  %95 Güven Aralığı: ({ci_low:.4f}, {ci_high:.4f})")
print(f"  Sonuç: {'H₀ REDDEDİLDİ ✓' if p_f < 0.05 else 'H₀ reddedilemedi'}  (p {'<' if p_f < 0.05 else '>='} 0.05)")

print("\n" + "=" * 60)
print("TEMEL FORMÜLLER")
print("=" * 60)
formüller = {
    "Eğim": "b₁ = (sy/sx) × R",
    "Kesim": "b₀ = ȳ - b₁×x̄",
    "Artık": "eᵢ = yᵢ - ŷᵢ",
    "R²": "R² = r²  (korelasyonun karesi)",
    "T istatistiği": "T = b₁ / SH_b₁",
    "Serbestlik Derecesi": "df = n - 2",
    "Güven Aralığı": "b₁ ± t*_(n-2) × SH_b₁",
}
for isim, formül in formüller.items():
    print(f"  {isim:<25} {formül}")

---
## Alıştırma Soruları

Aşağıdaki soruları kod yazarak yanıtlayın:

1. Lise mezunu oranı **%88** olan bir eyalet için yoksulluk oranını tahmin edin.
2. Sunumdaki modelde R² = 0.56 ise, açıklanamayan değişkenliğin payı nedir?
3. Aşağıdaki veri seti için basit doğrusal regresyon modeli oluşturun, katsayıları yorumlayın ve koşulları kontrol edin.

In [ ]:
# Alıştırma 1: Tahmin
lise_mez_alıştırma = 88
# Kodunuzu buraya yazın:
# tahmin = ...
# print(f"Tahmin: {tahmin:.2f}")

# Alıştırma 2: Açıklanamayan değişkenlik
R2_verilen = 0.56
# Kodunuzu buraya yazın:
# aciklanamayan = ...
# print(f"Açıklanamayan: %{aciklanamayan*100:.0f}")

# Alıştırma 3: Yeni veri seti
np.random.seed(99)
çalışma_saatleri = np.random.uniform(20, 60, 40)
sınav_notu = 30 + 1.2 * çalışma_saatleri + np.random.normal(0, 8, 40)
sınav_notu = np.clip(sınav_notu, 0, 100)

print("Alıştırma 3 verisi:")
print(f"  Çalışma saatleri: ort={çalışma_saatleri.mean():.1f}, ss={çalışma_saatleri.std():.1f}")
print(f"  Sınav notu:       ort={sınav_notu.mean():.1f}, ss={sınav_notu.std():.1f}")
print(f"  Korelasyon: {np.corrcoef(çalışma_saatleri, sınav_notu)[0,1]:.3f}")
print()
print("TODO: Bu veri için doğrusal regresyon modeli kurun ve yorumlayın.")